---
<div align="center">

# **ANÁLISIS EXPLORATORIO DE DATOS - Gestión del Fraude**

<div align="center">
  <img src="img/logo_uptc2.jpg" width="120">
</div>

**Profesor:** Duván Cataño  

**Curso:** Estadística para Analítica de Datos 

**Universidad Pedagógica y Tecnológica de Colombia**

</div>

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">


Este notebook sirve como guía base para el curso de Analítica de Datos. Incluye procesos de carga de datos y análisis exploratorio (EDA) aplicados a un conjunto de datos que contiene información Transaccional, Comportamiento Histórico, de Tiempo Relativo y de Perfil de individuos que han sido víctima de **fraude**. El objetivo principal es analizar el perfil de los clientes y predecir su riesgo a ser víctimas de fraude, el cual está representado por la variable objetivo (*FRAUDE*).

Cada registro corresponde a un individuo perfilado y contiene variables relacionadas con:

- Transaccionales

- Perfil del cliente

- Comportamiento histórico

- Tiempo relativo

- La variable **FRAUDE** representa la variable objetivo.

---

</div>


# Librerías

In [ ]:
# ================= #
# Cargar librearías #
# ================= #

import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import math
import plotly.graph_objects as go
import plotly.express as px

from scipy import stats
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")

# Carga de Datos

In [ ]:
# ================ #
# Configurar rutas #
# ================ #

mainpath= "/Users/duvancatano/Documents/Data_Analytics_UdeA/ml-project/data/fraud"
filename= "train.csv"
fullpath= os.path.join(mainpath,filename)

In [ ]:
# ================ #
# Lectura de datos #
# ================ #

data = pd.read_csv(fullpath, sep=";")      # El separador es "," porque en el archivo .csv los valores están separados por coma
pd.set_option('display.max_columns', None) # Para mostrar todas las columnas del DataFrame sin truncar

In [ ]:
# ============================ #
# Visualización de una Muestra #
# ============================ #

data.sample(20)

---

<div align="center"> 

| NOMBRE           | DESCRIPCIÓN                                                                 |
|------------------|-----------------------------------------------------------------------------|
| ID               | Id Cliente                                                                  |
| FRAUDE           | 1= Fraude; 0=No fraude                                                      |
| VALOR            | Valor de la transacción                                                     |
| HORA_AUX         | Hora de la transacción, sin minutos ni segundos                             |
| Dist_max_NAL     | Dist máxima recorrida a nivel nacional (en millas)                          |
| Canal1           | Canal transaccional de la transacción, incluido tipos de datafonos          |
| FECHA            | Fecha de ocurrencia de la transacción                                       |
| COD_PAIS         | País de ocurrencia de la transacción. Ver código ISO Internet               |
| CANAL            | Canal transaccional de la transacción                                       |
| DIASEM           | Día de la semana que se realizó la transacción (0= Domingo, 1= Lunes, ..., 6= sábado) |
| DIAMES           | Día del mes que se realizó la transacción  |
| FECHA_VIN        | Fecha de vinculación del cliente                                            |
| OFICINA_VIN      | Oficina de vinculación del cliente                                          |
| SEXO             | M=masculino, F=femenino                                                      |
| SEGMENTO         | Segmento del cliente                                                         |
| EDAD             | Edad del cliente                                                             |
| INGRESOS         | Ingresos del cliente                                                         |
| EGRESOS          | Egresos del cliente                                                          |
| NROPAISES        | Número de países visitados                                                   |
| Dist_Sum_INTER   | Sumatoria de distancia recorrida a nivel internacional (en millas)          |
| Dist_Mean_INTER  | Promedio de distancia recorrida a nivel internacional (en millas)           |
| NROCIUDADES      | Número de ciudades nacionales visitadas                                     |
| Dist_Sum_NAL     | Distancia máxima recorrida a nivel nacional (en millas)                     |
| Dist_Mean_NAL    | Distancia máxima recorrida a nivel nacional (en millas)                     |
| Dist_HOY         | Diferencia entre la última transacción presente realizada y la transacción que está realizando el día de hoy |
| Dist_sum_NAL     | Sumatoria de distancia recorrida a nivel nacional (en millas)               |

</div>

---

# **Definición del Problema**

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 

El aumento de las transacciones financieras digitales ha incrementado el riesgo de fraude, afectando tanto a las entidades como a la confianza en el sistema. Para enfrentar este problema, se dispone de una base de datos con información transaccional, geográfica y sociodemográfica de los clientes, donde la variable FRAUDE indica si una operación es fraudulenta.

Con el objetivo de enterder la información disponible en la base datos y realción entre las variables, son planteadas las siguientes preguntas:

</div>

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

**1.** ¿Qué tan desbalanceada está la variable FRAUDE? ¿Cuál es la proporción de transacciones fraudulentas vs no fraudulentas?

**2.** ¿Las transacciones de mayor valor presentan mayor probabilidad de fraude?

**3.** ¿El fraude ocurre más en ciertas horas del día? ¿Existen horarios críticos (por ejemplo, madrugada)?

**4.** ¿El fraude varía según el día de la semana (DIASEM)? ¿Hay días con mayor actividad fraudulenta?

**5.** ¿Existen países (COD_PAIS) con mayor incidencia de fraude? ¿Las transacciones internacionales presentan mayor riesgo?

**6.** ¿Qué canales (CANAL, Canal1) están más asociados al fraude? ¿Existen canales particularmente vulnerables?

**7.** ¿Clientes con mayor movilidad internacional (Dist_Sum_INTER, NROPAISES) presentan más fraude? ¿Distancias inusuales están asociadas a fraude?

**8.** ¿El fraude varía según edad, sexo o segmento? ¿Existen perfiles más propensos al fraude?

**9.** ¿La relación entre INGRESOS y EGRESOS influye en el fraude? ¿Clientes con desbalance financiero presentan mayor riesgo?

</div>

---

## **Objetivo General**

<div style="text-align: justify; line-height: 1.5; font-size: 18px;"> 
Identificar los factores y patrones asociados al fraude, analizando comportamientos atípicos y relaciones entre variables que permitan diferenciar transacciones fraudulentas de legítimas, sirviendo como base para futuros modelos predictivos y estrategias de mitigación del riesgo.

</div>

---

In [ ]:
# ======================= #
# Dimensiones del Dataset #
# ======================= #

data.shape

In [ ]:
# =========================================== #
# Mostrar las primeras 10 filas del DataFrame #
# =========================================== #

data.head(10) 

In [ ]:
# ========================================== #
# Mostrar las últimas 10 filas del DataFrame #
# ========================================== #

data.tail(10)

In [ ]:
# ===================== #
# Campos en una columna #
# ===================== #

data.columns.to_list()

In [ ]:
# ============================ #
# Información de las variables #
# ============================ #

data.info()

## **Funciones Pesonalizadas - Valores Perdidos y Nulos**

In [ ]:
# Porcentaje de valores perdidos por variable
def resum_missing(df):
    total = df.isnull().sum().sort_values(ascending=False)
    percent = (df.isnull().sum() *100/df.isnull().count()).sort_values(ascending=False)
    missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Porcentaje'])
    return missing_data

# Resumen de posición de valores perdidos
def posmissing(df):
    lista_miss = np.where(df.isna())
    v1 = pd.Series(np.ndarray.tolist(lista_miss[0]))
    v2 = pd.Series(np.ndarray.tolist(lista_miss[1]))
    resumen_vna = pd.concat([v1, v2], axis=1, keys=['posicion_fila', 'posicion_columna'])
    return resumen_vna

# Construcción función que cuenta ceros
def count_zeros(df):
    total = (df == 0).astype(int).sum(axis=0)
    percent = ((df == 0).astype(int).sum(axis=0) *100/df.count())
    missing_data = pd.concat([total, percent], axis=1, keys=['Total_ceros', 'Porcentaje'])
    return missing_data.sort_values(by='Porcentaje', ascending=False)

In [ ]:
# =========================================== #
# Porcentaje de valores perdidos por variable #
# =========================================== #

resum_missing(data)

In [ ]:
# ========================== #
# Porcentaje de valores cero #
# ========================== #

count_zeros(data)

In [ ]:
# ================================= #
# Densidades de Variables numéricas #
# ================================= #

# VARIABLES NUMÉRICAS
num_vars = data.select_dtypes(include=["int64", "float64"]).columns.tolist()

num_vars = [col for col in num_vars if col not in ["ID"]]

print("Variables disponibles:")
print(num_vars)

# VARIABLES SELECCIONADAS
vars_selected = [
    "VALOR",
    "EDAD",
    "NROPAISES",
    "Dist_Sum_INTER", 
    "Dist_Mean_INTER",
    "NROCIUDADES"
]

# SUBPLOTS
n_cols = 2
n_rows = math.ceil(len(vars_selected) / n_cols)

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[
        f"Distribución de {col}"
        for col in vars_selected
    ]
)

# COLORES ESTÉTICOS
colors = px.colors.qualitative.Pastel

# HISTOGRAMAS
for i, col in enumerate(vars_selected):

    row = (i // n_cols) + 1
    col_pos = (i % n_cols) + 1

    fig.add_trace(
        go.Histogram(
            x=data[col],
            nbinsx=40,
            marker=dict(
                color=colors[i % len(colors)],
                line=dict(width=1)
            ),
            opacity=0.85,
            name=col
        ),
        row=row,
        col=col_pos
    )

# LAYOUT
fig.update_layout(
    height=350 * n_rows,
    width=1100,
    template="plotly_dark",
    title="Distribuciones de Variables Numéricas",
    title_x=0.5,
    showlegend=False,
    bargap=0.08
)

# EJES
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True)
fig.show()

In [ ]:
# ================================================== #
# Distribuciones de Ingresos y Egresos en Escala Log #
# ================================================== #

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=[
        "INGRESOS (escala original)",
        "INGRESOS (escala log)",
        "EGRESOS (escala original)",
        "EGRESOS (escala log)"
    ]
)

# INGRESOS ORIGINAL
fig.add_trace(
    go.Histogram(
        x=data["INGRESOS"],
        nbinsx=40,
        marker=dict(color="#6EC6FF"),
        opacity=0.85
    ),
    row=1,
    col=1
)

# INGRESOS LOG
fig.add_trace(
    go.Histogram(
        x=np.log1p(data["INGRESOS"]),
        nbinsx=40,
        marker=dict(color="#81C784"),
        opacity=0.85
    ),
    row=1,
    col=2
)

# EGRESOS ORIGINAL
fig.add_trace(
    go.Histogram(
        x=data["EGRESOS"],
        nbinsx=40,
        marker=dict(color="#FFB74D"),
        opacity=0.85
    ),
    row=2,
    col=1
)

# EGRESOS LOG
fig.add_trace(
    go.Histogram(
        x=np.log1p(data["EGRESOS"]),
        nbinsx=40,
        marker=dict(color="#FF8A80"),
        opacity=0.85
    ),
    row=2,
    col=2
)

# LAYOUT
fig.update_layout(
    height=750,
    width=1100,
    template="plotly_dark",
    title="Distribuciones de Ingresos y Egresos",
    title_x=0.5,
    showlegend=False,
    bargap=0.08
)

# EJES
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True)
fig.show()

## **Imputación**

In [ ]:
from sklearn.impute import KNNImputer

data_imp = data.copy()

# MODAS GLOBALES
sexo_mode = data_imp["SEXO"].mode()[0]
segmento_mode = data_imp["SEGMENTO"].mode()[0]

# IMPUTACIÓN BASE
data_imp["SEXO"] = data_imp["SEXO"].fillna(sexo_mode)
data_imp["SEGMENTO"] = data_imp["SEGMENTO"].fillna(segmento_mode)

# SEXO
mask_sexo = data["SEXO"].isna()

sexo_condicional = (
    data_imp.groupby("SEGMENTO")["SEXO"]
    .agg(lambda x: x.mode()[0])
)

data_imp.loc[mask_sexo, "SEXO"] = (
    data_imp.loc[mask_sexo, "SEGMENTO"]
    .map(sexo_condicional)
)

# SEGMENTO
mask_segmento = data["SEGMENTO"].isna()

segmento_condicional = (
    data_imp.groupby("SEXO")["SEGMENTO"]
    .agg(lambda x: x.mode()[0])
)

data_imp.loc[mask_segmento, "SEGMENTO"] = (
    data_imp.loc[mask_segmento, "SEXO"]
    .map(segmento_condicional)
)

# KNN SOLO NUMÉRICAS
vars_knn = [
    "EGRESOS",
    "INGRESOS",
    "FECHA_VIN",
    "OFICINA_VIN",
    "EDAD"
]

data_knn = data_imp[vars_knn]

imputer = KNNImputer(n_neighbors=5)

data_knn_imputed = imputer.fit_transform(data_knn)

data_imp[vars_knn] = pd.DataFrame(
    data_knn_imputed,
    columns=vars_knn,
    index=data_imp.index
)

# VARIABLES A CERO
vars_zero = [
    "Dist_Max_INTER",
    "Dist_Mean_INTER",
    "Dist_Sum_INTER",
    "Dist_Mean_NAL"
]

data_imp[vars_zero] = data_imp[vars_zero].fillna(0)

# CONTEOS
print(data_imp["SEGMENTO"].value_counts(dropna=False))
print(data_imp["SEXO"].value_counts(dropna=False))

In [ ]:
# ========================== #
# Verificando la Imptucación #
# ========================== #

resum_missing(data_imp)

## **Análisis de Valores Atípicos en EDAD**

In [ ]:
from sklearn.impute import KNNImputer

data_full = data_imp.copy()

# Definición: edad inválida
cond_outliers = (data_full["EDAD"] <= 0) | (data_full["EDAD"] > 100)

# Convertir a NaN
data_full.loc[cond_outliers, "EDAD"] = None

# Variables para KNN #
vars_knn = [
    "EDAD",
    "INGRESOS",
    "EGRESOS",
    "NROPAISES",
    "NROCIUDADES",
    "Dist_HOY"
]

data_knn = data_full[vars_knn].copy()

# Aplicar KNN #
imputer = KNNImputer(n_neighbors=5)

data_knn_imputed = imputer.fit_transform(data_knn)

data_knn_imputed = pd.DataFrame(
    data_knn_imputed,
    columns=vars_knn,
    index=data_full.index
)

# Reemplazar EDAD imputada
data_imp["EDAD"] = data_knn_imputed["EDAD"]

In [ ]:
# ======================================= #
# Gráfico de la Densidad de EDAD imputada #
# ======================================= #

import plotly.figure_factory as ff

# DENSIDAD DE LA VARIABLE EDAD
edad = data_full["EDAD"].dropna()

# KDE INTERACTIVO
fig = ff.create_distplot(
    [edad],
    group_labels=["EDAD"],
    show_hist=False,
    show_rug=False
)

# ESTÉTICA
fig.update_traces(
    line=dict(color="#6EC6FF", width=3)
)

fig.update_layout(
    height=500,
    width=850,
    template="plotly_dark",
    title="Densidad de la Variable EDAD",
    title_x=0.5,
    xaxis_title="Edad",
    yaxis_title="Densidad",
    showlegend=False
)

# LÍMITE EJE X
fig.update_xaxes(range=[0, 90])
fig.show()

In [ ]:
# =============================== #
# Nombrando los Días de la Semana #
# =============================== #

map_dias = {
    0: "Domingo",
    1: "Lunes",
    2: "Martes",
    3: "Miercoles",
    4: "Jueves",
    5: "Viernes",
    6: "Sabado"
}

# Reemplazar en la columna
data_full["DIASEM"] = data_full["DIASEM"].map(map_dias)

In [ ]:
# ======================================== #
# Creando quincenas con la variable DIAMES #
# ======================================== #

data_full["QUINCENA"] = pd.Series(index=data_full.index, dtype="object")

data_full.loc[data_full["DIAMES"] <= 15, "QUINCENA"] = "Quincena_1"
data_full.loc[data_full["DIAMES"] > 15, "QUINCENA"] = "Quincena_2"

In [ ]:
# ============================ #
# Vista de los Datos Completos #
# ============================ #

data_full.sample(20)

---

## **Respondiendo Posibles Hipótesis**

### **1. Balance de FRAUDE**

In [ ]:
import plotly.express as px

# TABLA RESUMEN
conteo = data_full["FRAUDE"].value_counts().sort_index()
prop = data_full["FRAUDE"].value_counts(normalize=True).sort_index()

tabla = pd.DataFrame({
    "Conteo": conteo,
    "Proporción": prop
})

print(tabla)

# DATAFRAME PARA PLOTLY
df_plot = pd.DataFrame({
    "Clase": ["No fraude (0)", "Fraude (1)"],
    "Conteo": conteo.values,
    "Proporcion": prop.values
})

# GRÁFICA ESTÉTICA
fig = px.bar(
    df_plot,
    x="Clase",
    y="Conteo",
    color="Proporcion",
    color_continuous_scale="Reds",
    text=df_plot["Proporcion"].apply(lambda x: f"{x*100:.2f}%"),
    title="Distribución de FRAUDE"
)

# ESTILO
fig.update_traces(
    textposition="outside"
)

fig.update_layout(
    height=500,
    width=700,
    template="plotly_dark",
    xaxis_title="Clase",
    yaxis_title="Número de transacciones",
    showlegend=False,
    title_x=0.5
)

fig.show()

## **2. Valor vs Fraude**

In [ ]:
# =========================
# BOXPLOT VALOR vs FRAUDE
# =========================

fig = px.box(
    data_full,
    x="FRAUDE",
    y="VALOR",
    color="FRAUDE",
    points="outliers",
    title="Valor de Transacción vs Fraude"
)

# ESCALA LOG
fig.update_yaxes(type="log")


# ESTÉTICA
fig.update_layout(
    height=500,
    width=750,
    template="plotly_dark",
    xaxis_title="Fraude",
    yaxis_title="Valor (escala log)",
    showlegend=False,
    title_x=0.5
)

fig.show()

### **3. Hora vs Fraude**

In [ ]:
# =========================
# TABLA DE PROPORCIONES
# =========================

tabla = pd.crosstab(
    data_full["HORA_AUX"],
    data_full["FRAUDE"],
    normalize="index"
)

tabla.columns = ["No fraude", "Fraude"]

tabla = tabla.reset_index()

# GRÁFICA INTERACTIVA
fig = px.line(
    tabla,
    x="HORA_AUX",
    y="Fraude",
    markers=True,
    title="Probabilidad de Fraude por Hora"
)

fig.update_traces(
    line=dict(width=3)
)

fig.update_layout(
    height=500,
    width=850,
    template="plotly_dark",
    xaxis_title="Hora del día",
    yaxis_title="Proporción de fraude",
    title_x=0.5
)

fig.show()

### **4. Día de la semana**

In [ ]:
# ORDEN DE LOS DÍAS
orden_dias = [
    "Domingo",
    "Lunes",
    "Martes",
    "Miercoles",
    "Jueves",
    "Viernes",
    "Sabado"
]

data_full["DIASEM"] = pd.Categorical(
    data_full["DIASEM"],
    categories=orden_dias,
    ordered=True
)

# TABLA DE PROPORCIONES
tabla = pd.crosstab(
    data_full["DIASEM"],
    data_full["FRAUDE"],
    normalize="index"
)

tabla.columns = ["No fraude", "Fraude"]

tabla = tabla.reset_index()

# GRÁFICA INTERACTIVA
fig = px.bar(
    tabla,
    x="DIASEM",
    y="Fraude",
    color="Fraude",
    color_continuous_scale="Reds",
    text_auto=".2f",
    title="Probabilidad de Fraude por Día de la Semana"
)

# ESTÉTICA
fig.update_layout(
    height=500,
    width=850,
    template="plotly_dark",
    xaxis_title="Día de la semana",
    yaxis_title="Proporción de fraude",
    showlegend=False,
    title_x=0.5
)

fig.show()

### **5. País**

In [ ]:
# TABLA DE PROPORCIONES
tabla = pd.crosstab(
    data_full["COD_PAIS"],
    data_full["FRAUDE"],
    normalize="index"
)

tabla.columns = ["No fraude", "Fraude"]

# Ordenar por mayor fraude
tabla = tabla.sort_values(by="Fraude", ascending=False)

# Opcional: Top 15 países
tabla = tabla.head(15)

tabla = tabla.reset_index()

# GRÁFICA INTERACTIVA
fig = px.bar(
    tabla,
    x="COD_PAIS",
    y="Fraude",
    color="Fraude",
    color_continuous_scale="Reds",
    text_auto=".2f",
    title="Top Países con Mayor Proporción de Fraude"
)

fig.update_layout(
    height=550,
    width=950,
    template="plotly_dark",
    xaxis_title="Código País",
    yaxis_title="Proporción de fraude",
    showlegend=False,
    title_x=0.5
)

fig.update_xaxes(tickangle=45)

fig.show()

### **6. Canales**

In [ ]:
# TABLA DE PROPORCIONES
tabla = pd.crosstab(
    data_full["CANAL"],
    data_full["FRAUDE"],
    normalize="index"
)

tabla.columns = ["No fraude", "Fraude"]

# Ordenar por mayor proporción de fraude
tabla = tabla.sort_values(by="Fraude", ascending=False)

tabla = tabla.reset_index()

# GRÁFICA INTERACTIVA
fig = px.bar(
    tabla,
    x="CANAL",
    y="Fraude",
    color="Fraude",
    color_continuous_scale="Reds",
    text_auto=".2f",
    title="Proporción de Fraude por Canal"
)

fig.update_layout(
    height=500,
    width=900,
    template="plotly_dark",
    xaxis_title="Canal",
    yaxis_title="Proporción de fraude",
    showlegend=False,
    title_x=0.5
)

fig.update_xaxes(tickangle=35)

fig.show()

### **7. Movilidad Internacional**

In [ ]:
# SCATTER MOVILIDAD vs FRAUDE #
fig = px.scatter(
    data_full,
    x="Dist_Sum_INTER",
    y="NROPAISES",
    color="FRAUDE",
    opacity=0.65,
    title="Movilidad Internacional vs Fraude",
    hover_data=[
        "VALOR",
        "CANAL",
        "COD_PAIS"
    ],

    # Colores más claros
    color_discrete_map={
        0: "#6EC6FF",   # azul claro
        1: "#FF8A80"    # rojo claro
    }
)

fig.update_layout(
    height=550,
    width=850,
    template="plotly_dark",
    xaxis_title="Distancia internacional acumulada",
    yaxis_title="Número de países visitados",
    title_x=0.5
)

fig.show()

### **8. Perfil del cliente**

In [ ]:
# EDAD 
plt.style.use("dark_background")

# KDE EDAD vs FRAUDE
plt.figure(figsize=(8,5))

sns.kdeplot(
    data=data_full,
    x="EDAD",
    hue="FRAUDE",
    fill=True,
    alpha=0.4,
    linewidth=2,
    palette=["#6EC6FF", "#FF8A80"]  # colores claros
)

plt.title(
    "Distribución de Edad por Fraude",
    fontsize=16,
    weight="bold"
)

plt.xlabel("Edad", fontsize=12)
plt.ylabel("Densidad", fontsize=12)

plt.grid(alpha=0.15)

plt.tight_layout()

plt.show()

In [ ]:
# SEXO 
tabla = pd.crosstab(
    data_full["SEXO"],
    data_full["FRAUDE"],
    normalize="index"
)

tabla.columns = ["No fraude", "Fraude"]

tabla = tabla.reset_index()

# GRÁFICA INTERACTIVA
fig = px.bar(
    tabla,
    x="SEXO",
    y="Fraude",
    color="Fraude",
    color_continuous_scale="Reds",
    text_auto=".2f",
    title="Proporción de Fraude por Sexo"
)

fig.update_layout(
    height=500,
    width=750,
    template="plotly_dark",
    xaxis_title="Sexo",
    yaxis_title="Proporción de fraude",
    showlegend=False,
    title_x=0.5
)

fig.show()

### **9. Ingresos vs Egresos**

In [ ]:
# SCATTER INGRESOS vs EGRESOS
fig = px.scatter(
    data_full,
    x="INGRESOS",
    y="EGRESOS",
    color="FRAUDE",
    opacity=0.6,
    title="Ingresos vs Egresos",

    # Colores claros
    color_discrete_map={
        0: "#6EC6FF",   # azul claro
        1: "#FF8A80"    # rojo claro
    },

    hover_data=[
        "SEXO",
        "SEGMENTO",
        "EDAD",
        "CANAL"
    ]
)

# ESCALAS LOG
fig.update_xaxes(type="log")
fig.update_yaxes(type="log")

fig.update_layout(
    height=550,
    width=850,
    template="plotly_dark",
    xaxis_title="Ingresos (escala log)",
    yaxis_title="Egresos (escala log)",
    title_x=0.5
)

fig.show()

### **10. Segmento vs Ingresos**

In [ ]:
# SEGMENTO CON INGRESOS
fig = px.box(
    data_full,
    x="SEGMENTO",
    y="INGRESOS",
    color="SEGMENTO",
    points="outliers",
    title="Distribución de Ingresos por Segmento"
)

# Escala log
fig.update_yaxes(type="log")

# Layout
fig.update_layout(
    height=500,
    xaxis_title="Segmento",
    yaxis_title="Ingresos (escala log)",
    showlegend=False,
    template="plotly_dark",

    title_x=0.5
)

fig.show()

---

### **Guardar la Base de Datos Procesada**

In [ ]:
# ============================== #
# Guardar Base de Datos Completa #
# ============================== #

ruta_dir = "/Users/duvancatano/Documents/Estadistica_Analitica_Datos_UTPC_2026-1/data/fraud"
os.makedirs(ruta_dir, exist_ok=True)

ruta_file = os.path.join(ruta_dir, "data_full.csv")

data_full.to_csv(ruta_file, index=False)

---

### Para generar el Streamlit - Ejecución desde la Terminal

```streamlit run src/fraud_eda.py```


---

# 🎬 **¡FIN!**

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">    </div>